# Preprocess data for LLMs

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    print("Total number of character:", len(raw_text))
    print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## Tokenize the text

In [2]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
result = re.split(r'([,.]|\s)', text)
print(result)


['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [4]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [5]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)


['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [6]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))


4690


In [7]:
print(preprocessed[:30])


['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Convert tokens to token IDs

In [10]:
words = sorted(set(preprocessed))
vocab_size = len(words)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1130


In [12]:
vocab = {word: i for i, word in enumerate(words)}
for token, token_id in vocab.items():
    print(f"{token}: {token_id}")

!: 0
": 1
': 2
(: 3
): 4
,: 5
--: 6
.: 7
:: 8
;: 9
?: 10
A: 11
Ah: 12
Among: 13
And: 14
Are: 15
Arrt: 16
As: 17
At: 18
Be: 19
Begin: 20
Burlington: 21
But: 22
By: 23
Carlo: 24
Chicago: 25
Claude: 26
Come: 27
Croft: 28
Destroyed: 29
Devonshire: 30
Don: 31
Dubarry: 32
Emperors: 33
Florence: 34
For: 35
Gallery: 36
Gideon: 37
Gisburn: 38
Gisburns: 39
Grafton: 40
Greek: 41
Grindle: 42
Grindles: 43
HAD: 44
Had: 45
Hang: 46
Has: 47
He: 48
Her: 49
Hermia: 50
His: 51
How: 52
I: 53
If: 54
In: 55
It: 56
Jack: 57
Jove: 58
Just: 59
Lord: 60
Made: 61
Miss: 62
Money: 63
Monte: 64
Moon-dancers: 65
Mr: 66
Mrs: 67
My: 68
Never: 69
No: 70
Now: 71
Nutley: 72
Of: 73
Oh: 74
On: 75
Once: 76
Only: 77
Or: 78
Perhaps: 79
Poor: 80
Professional: 81
Renaissance: 82
Rickham: 83
Riviera: 84
Rome: 85
Russian: 86
Sevres: 87
She: 88
Stroud: 89
Strouds: 90
Suddenly: 91
That: 92
The: 93
Then: 94
There: 95
They: 96
This: 97
Those: 98
Though: 99
Thwing: 100
Thwings: 101
To: 102
Usually: 103
Venetian: 104
Victor: 105
Was: 1

In [13]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {token_id:token for token, token_id in vocab.items()}

    def encode(self, text): #C
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, token_ids):
        text = " ".join([self.int_to_str[token_id] for token_id in token_ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # Replace spaces before the specified punctuation

        return text

In [14]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable
pride."""
ids = tokenizer.encode(text)
print(ids)


[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [15]:
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [16]:
text = "Hello, do you like tea?"
print(tokenizer.encode(text))


KeyError: 'Hello'

## Adding special context tokens

In [18]:
words = sorted(set(preprocessed))
words.extend(["<|endoftext|>", "<|unk|>"])
vocab = {word: i for i, word in enumerate(words)}

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [20]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {token_id:token for token, token_id in vocab.items()}

    def encode(self, text): #C
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else '<|unk|>' for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, token_ids):
        text = " ".join([self.int_to_str[token_id] for token_id in token_ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # Replace spaces before the specified punctuation

        return text

In [21]:
tokenizerv2 = SimpleTokenizerV2(vocab)
text = "Hello, do you like tea?"
print(tokenizerv2.encode(text))


[1131, 5, 355, 1126, 628, 975, 10]


In [24]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)
print(tokenizerv2.encode(text))


Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [26]:
print(tokenizerv2.decode(tokenizerv2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## Byte pair encoding

In [29]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [30]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))


tiktoken version: 0.12.0


In [32]:
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)


[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [34]:
text = "Akwirw ier"
integers = tokenizer.encode(text)
print(integers)

strings = tokenizer.decode(integers)
print(strings)


[33901, 86, 343, 86, 220, 959]
Akwirw ier


## Data sampling with a sliding window

In [36]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    enc_text = tokenizer.encode(raw_text)
    print(len(enc_text))

enc_sample = enc_text[50:]


5145


In [41]:
context_size = 4 #A
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")


x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [43]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
    print()


[290] ----> 4920
 and ---->  established

[290, 4920] ----> 2241
 and established ---->  himself

[290, 4920, 2241] ----> 287
 and established himself ---->  in

[290, 4920, 2241, 287] ----> 257
 and established himself in ---->  a



In [50]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [51]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)
        number_of_tokens = len(token_ids)

        for i in range(0, number_of_tokens-max_length, stride):
            self.input_ids.append(
                torch.tensor(token_ids[i:i+max_length])
                )
            self.target_ids.append(
                torch.tensor(token_ids[i+1:i+max_length+1])
                )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

c:\Users\a324a284.st\AppData\Local\anaconda3\envs\llm_from_scratch\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [52]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2") 
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) 
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=0
    )
    return dataloader

In [53]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
    data_iter = iter(dataloader)
    first_batch = next(data_iter)
    print(first_batch)


[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
# Shuffle off
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=2, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

# Shuffle on
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=2, shuffle=True)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 2885,  1464,  1807,  3619],
        [ 1807,  3619,   402,   271],
        [  402,   271, 10899,  2138],
        [10899,  2138,   257,  7026],
        [  257,  7026, 15632,   438],
        [15632,   438,  2016,   257],
        [ 2016,   257,   922,  5891]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 1464,  1807,  3619,   402],
        [ 3619,   402,   271, 10899],
        [  271, 10899,  2138,   257],
        [ 2138,   257,  7026, 15632],
        [ 7026, 15632,   438,  2016],
        [  438,  2016,   257,   922],
        [  257,   922,  5891,  1576]])
Inputs:
 tensor([[ 1640,   340,   373,   407],
        [ 7163,   262,  8631, 26210],
        [   13,   314, 11856,   510],
        [ 2497,   326,   339,   373],
        [  353,   438,  2934,   489],
        [  425,   284,  3465,   644],
        [  465,  2426,    11, 19233],
        [  257,  1049,  1517,    13]])

Targets:
 tensor([[  340,   373,   407, 10597],
       

## Token Embeddings

In [59]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))


Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## Encoding word positions

In [63]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)


max_length = 4
dataloader = create_dataloader_v1(
raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

token_embeddings = token_embedding_layer(inputs)
print("token_embeddings shape", token_embeddings.shape)

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("pos_embedding_layer shape", pos_embedding_layer.weight.shape)
print("pos_embeddings shape", pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print("input_embeddings shape", input_embeddings.shape)




Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])
token_embeddings shape torch.Size([8, 4, 256])
pos_embedding_layer shape torch.Size([4, 256])
pos_embeddings shape torch.Size([4, 256])
input_embeddings shape torch.Size([8, 4, 256])
